In [0]:
catalog = "weather_de"
schema = "weather"

spark.sql(f"use catalog {catalog}")
spark.sql(f"use schema {schema}")

DataFrame[]

Calling the API and uploading in the table

In [0]:

import requests
import json
from datetime import datetime, timezone, timedelta
import pandas as pd
from delta.tables import DeltaTable
from pyspark.sql import functions as f




In [0]:
today = datetime.now().date()
end_date = (today + timedelta(days=14)).strftime("%Y-%m-%d")

if not spark.catalog.tableExists(f"{catalog}.{schema}.weather_hourly"):
    start_date = (today - timedelta(days=92)).strftime("%Y-%m-%d")
    print(f"Initial load: Fetching full 3-month history starting {start_date} to {end_date}")
else: #daily runs
    start_date = (today - timedelta(days=2)).strftime("%Y-%m-%d")
    print(f"Incremental update: Fetching new updated data from {start_date} to {end_date}")
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 52.52,
    "longitude": 13.41,
    "daily": ["weather_code", "sunrise", "sunset", "daylight_duration", "uv_index_max", "moonset", "moon_phase", "moonrise"],
    "hourly": ["temperature_2m", "precipitation_probability", "snowfall", "visibility", "pressure_msl", "soil_temperature_0cm", "uv_index", "is_day"],
    "timezone": "Europe/Berlin",
    "past_days":1,
    "forecast_days":14
}

# Python automatically combines the url + params for you
response = requests.get(url, params=params)
data = response.json()


Incremental update: Fetching new updated data from 2026-11-21 to 2026-12-07


Historization of tables

In [0]:
spark_daily = spark.createDataFrame(pd.DataFrame(data["hourly"]))

def historize_table(new_df, table_name):
    if not spark.catalog.tableExists(table_name):
        new_df.write.format("delta").saveAsTable(table_name)
    else:
        target = DeltaTable.forName(spark, table_name)
        (
            target.alias("t")
            .merge(new_df.alias("s"), "t.time = s.time")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

historize_table(spark_daily, f"{catalog}.{schema}.weather_hourly")
print("Historization complete")

Historization complete
